# WB2 — AI 이미지·포스터 제작 (student)

CPU, API 키 불필요. 독립 실습. 먼저 Drive 사본을 만드세요. AI 서비스에서 기본·개선 프롬프트를 각각 실행하고 실제 PNG/JPG를 저장합니다. 서비스 사용 불가 시 도형 배경으로 실습하며 생성하지 않았다고 기록합니다.

## 코드 셀 1. 데이터 읽기

**코드 설명:** read_csv 함수는 로컬 파일→GitHub→내장 데이터 순으로 읽습니다. DataFrame은 행과 열의 표입니다.

**체크포인트:** 4행 출력

In [ ]:
import pandas as pd, io, json
from pathlib import Path
from urllib.request import urlopen
from IPython.display import display, HTML
def read_csv(name, fallback):
    p = Path(name)
    if p.exists():
        raw = p.read_text(encoding="utf-8-sig")
    else:
        try:
            raw = urlopen("https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/week13/"+name, timeout=6).read().decode("utf-8-sig")
        except Exception:
            raw = fallback
    return pd.read_csv(io.StringIO(raw))
events = read_csv("event_briefs.csv", 'id,event,date,time,place,capacity,fee,deadline,audience,action\nA01,캠퍼스 북교환회,2026-12-03,14:00~16:00,학생회관 1층,40,무료,2026-12-01,재학생,도서 1권 지참 후 신청\nA02,디지털 사진 전시회,2026-12-04,13:00~17:00,도서관 로비,60,무료,2026-12-02,재학생,온라인 신청\nA03,텀블러 데이,2026-12-07,12:00~14:00,학생식당 앞,80,무료,2026-12-05,재학생,개인 텀블러 지참\nA04,진로 포트폴리오 특강,2026-12-08,15:00~17:00,강의동 201호,30,무료,2026-12-06,재학생,온라인 신청\n')
assert len(events) == 4 and events["id"].is_unique
display(events)


## 코드 셀 2. 이미지 프롬프트

**코드 설명:** loc로 행사 선택, f-string으로 행사명을 삽입합니다. 두 지시문을 새 채팅에 입력해 구성과 여백을 비교하세요.

**체크포인트:** 프롬프트 2개 출력

In [ ]:
event_id = "A01"
row = events.loc[events["id"] == event_id].iloc[0].to_dict()
basic = f"{row['event']} 홍보 포스터 배경"
improved = f"""행사: {row['event']}. 대학생 대상, 따뜻한 일러스트,
세로 4:5, 상단에 제목 여백, 하단 정보 영역 여백,
문자·상표·실존 인물 없이 배경만 생성."""
print("기본:",basic)
print("개선:",improved)


## 코드 셀 3. 포스터 합성

**코드 설명:** 파일 창에 개선 이미지 PNG/JPG를 업로드해 background_file에 이름을 입력하세요. base64는 이미지 바이트를 문자열로 바꿉니다. escape는 XML 특수문자를 안전하게 표시합니다. SVG는 좌표로 텍스트와 도형을 배치합니다. 비워두면 도형 배경입니다.

**체크포인트:** wb2_poster.svg 생성; 날짜·장소 원자료와 일치

In [ ]:
from html import escape
import base64
def safe(x): return escape(str(x), quote=True)
# 실제 생성된 PNG/JPG를 Colab 파일 창에 올리고 파일 이름을 입력합니다.
# 비워 두면 직접 만든 도형 배경(생성 AI가 아님)을 사용합니다.
background_file = ""
background = ""
if background_file:
    p = Path(background_file)
    assert p.is_file() and p.suffix.lower() in (".png",".jpg",".jpeg"), "PNG/JPG 파일을 먼저 업로드하세요."
    mime = "image/png" if p.suffix.lower()==".png" else "image/jpeg"
    encoded = base64.b64encode(p.read_bytes()).decode()
    background = f'<image href="data:{mime};base64,{encoded}" x="0" y="0" width="680" height="850" preserveAspectRatio="xMidYMid slice"/>'
else:
    background = '<rect width="680" height="850" fill="#dceee5"/><circle cx="340" cy="290" r="155" fill="#aed6c7"/><path d="M200 240h125v160H200z M355 240h125v160H355z" fill="#477f78"/>'
svg = f"""<svg xmlns="http://www.w3.org/2000/svg" width="680" height="850">
{background}
<rect x="25" y="30" width="630" height="145" rx="12" fill="#fff" opacity=".94"/>
<rect x="25" y="480" width="630" height="340" rx="12" fill="#fff" opacity=".96"/>
<g font-family="sans-serif" text-anchor="middle" fill="#183b35">
<text x="340" y="88" font-size="36" font-weight="bold">{safe(row['event'])}</text>
<text x="340" y="137" font-size="23">교육용 가상 행사</text>
<text x="340" y="535" font-size="25">{safe(row['date'])} · {safe(row['time'])}</text>
<text x="340" y="585" font-size="25">{safe(row['place'])}</text>
<text x="340" y="635" font-size="25">{safe(row['audience'])} · 정원 {safe(row['capacity'])}명 · {safe(row['fee'])}</text>
<text x="340" y="685" font-size="25">마감 {safe(row['deadline'])}</text>
<text x="340" y="745" font-size="25" font-weight="bold">{safe(row['action'])}</text>
</g></svg>"""
Path("wb2_poster.svg").write_text(svg,encoding="utf-8")
display(HTML(svg))
print("저장: wb2_poster.svg")


## 코드 셀 4. 기록 및 제출

**코드 설명:** 딕셔너리에 실제 도구, 날짜, 두 이미지 관찰, 사실 검토를 입력하고 JSON 저장. 파일 창에서 SVG와 JSON 및 원본 생성 이미지를 다운로드합니다.

**체크포인트:** wb2_record.json 생성

In [ ]:
record = {'tool_and_date': '', 'human_draft': '', 'basic_result_observation': '', 'improved_result_observation': '', 'fact_review': ''}
Path("wb2_record.json").write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding="utf-8")
print("저장: wb2_record.json — 학생은 기록 칸을 채우세요.")
